# LLM-Based RAG Evaluation
> Use an LLM judge to score faithfulness, relevancy, and context quality.

`LLMRAGEvaluator` wraps a custom evaluation function that returns
RAG-specific metrics. This lets you plug in any LLM-as-judge
implementation while keeping a consistent interface.

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor.evaluation import LLMRAGEvaluator
from anchor.evaluation.models import RAGMetrics

## Define a Mock Evaluation Function
In production you would call an LLM here. For this notebook we
return deterministic scores so the output is reproducible.

In [ ]:
def mock_judge(query, answer, contexts, ground_truth=None):
    """Simulate an LLM judge scoring a RAG response."""
    return RAGMetrics(
        faithfulness=0.9,
        relevancy=0.85,
        context_precision=0.8,
        context_recall=0.75,
    )

print("Evaluation function defined.")
print(f"Return type: {RAGMetrics.__name__}")

## Create the Evaluator
Pass the callable into `LLMRAGEvaluator`. It will be invoked
every time `.evaluate()` is called.

In [ ]:
evaluator = LLMRAGEvaluator(eval_fn=mock_judge)

print(f"Evaluator type: {type(evaluator).__name__}")
print(f"Has eval_fn:    {evaluator.eval_fn is not None}")

## Evaluate a Single Response
Provide the query, generated answer, supporting contexts, and
(optionally) a ground-truth reference answer.

In [ ]:
metrics = evaluator.evaluate(
    query="What is prompt engineering?",
    answer="Prompt engineering is the practice of designing inputs to LLMs.",
    contexts=["Prompt engineering involves crafting effective prompts for AI models."],
    ground_truth="Prompt engineering is the systematic design of LLM inputs.",
)

print(f"Faithfulness:      {metrics.faithfulness:.2f}")
print(f"Relevancy:         {metrics.relevancy:.2f}")
print(f"Context Precision: {metrics.context_precision:.2f}")
print(f"Context Recall:    {metrics.context_recall:.2f}")

## Evaluate Without Ground Truth
Ground truth is optional -- useful when you only want to assess
faithfulness and relevancy.

In [ ]:
metrics_no_gt = evaluator.evaluate(
    query="What is RAG?",
    answer="RAG stands for Retrieval-Augmented Generation.",
    contexts=["RAG combines retrieval with generation to improve accuracy."],
)

print(f"Faithfulness:      {metrics_no_gt.faithfulness:.2f}")
print(f"Relevancy:         {metrics_no_gt.relevancy:.2f}")
print(f"Context Precision: {metrics_no_gt.context_precision:.2f}")
print(f"Context Recall:    {metrics_no_gt.context_recall:.2f}")

## Custom Scoring Functions
You can vary the mock to simulate different quality levels.

In [ ]:
def low_quality_judge(query, answer, contexts, ground_truth=None):
    return RAGMetrics(
        faithfulness=0.3,
        relevancy=0.25,
        context_precision=0.4,
        context_recall=0.2,
    )

low_evaluator = LLMRAGEvaluator(eval_fn=low_quality_judge)
low_metrics = low_evaluator.evaluate(
    query="What is X?",
    answer="X is something.",
    contexts=["Unrelated context."],
)

print("Low-quality response scores:")
print(f"  Faithfulness:      {low_metrics.faithfulness:.2f}")
print(f"  Relevancy:         {low_metrics.relevancy:.2f}")
print(f"  Context Precision: {low_metrics.context_precision:.2f}")
print(f"  Context Recall:    {low_metrics.context_recall:.2f}")

## Key Takeaways

- `LLMRAGEvaluator` accepts any callable that returns `RAGMetrics`.
- Four dimensions are captured: faithfulness, relevancy,
  context precision, and context recall.
- Ground truth is optional; omit it for pure faithfulness checks.
- Swap the scoring function to upgrade from a mock to a real LLM judge.